# Cenário do Projeto: Controle de Estoque

**Descrição do Cenário:** 
O cenário abordado neste projeto simula um sistema simplificado de **Controle de Estoque** de uma loja de eletrônicos. A tabela principal armazena informações vitais dos produtos, como ID, nome do produto, categoria, quantidade disponível e preço unitário. 

**Conjunto de Dados (Dataset):**
O conjunto de dados utilizado foi **criado manualmente (Mock Data)** diretamente via código no Apache Spark para fins de demonstração das operações ACID (Insert, Update, Delete) nos formatos Delta Lake e Apache Iceberg.

**Modelo Entidade-Relacionamento (ER):**
Abaixo está a representação da nossa entidade `estoque_delta`.

```mermaid
erDiagram
    ESTOQUE {
        int id PK
        string nome_produto
        string categoria
        int quantidade
        float preco
    }
```

*(Adicione uma imagem ilustrativa do banco de dados na pasta e referencie aqui: `![Modelo ER](./imagem_estoque.png)`)*

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, FloatType
from delta import *
import logging

logging.getLogger("py4j").setLevel(logging.DEBUG)

# Inicializando a SparkSession com o formato Delta
spark = (
    SparkSession
    .builder
    .master("local[*]")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)
spark

: 

### DDL - Criação da Tabela
Aqui executamos o comando Data Definition Language (DDL) para criar a estrutura da tabela do nosso estoque. Definimos as colunas iniciais e o formato da tabela indicando `USING delta`.

In [ ]:
spark.sql("""
  CREATE TABLE IF NOT EXISTS estoque_delta (id INT, nome_produto STRING) USING delta
""")

spark.sql("select * from estoque_delta").show()

### INSERT - Inserção de Dados
O comando `INSERT INTO` é utilizado para popular a tabela com nossos produtos iniciais. Este comando demonstra a capacidade de escrita transacional no formato Delta.

In [ ]:
spark.sql("INSERT INTO estoque_delta VALUES (1, 'Notebook Dell'), (2, 'Mouse Logitech'), (3, 'Teclado Mecânico')")

spark.sql("select * from estoque_delta").show()

### UPDATE - Atualização de Dados
Utilizamos o comando `UPDATE` para alterar registros existentes no estoque. Primeiro vamos evoluir o schema adicionando novas colunas (categoria, quantidade e preco) e depois atualizá-las evidenciando o suporte a operações ACID a nível de linha.

In [ ]:
# Adicionando as novas colunas
spark.sql("""
    ALTER TABLE estoque_delta ADD COLUMN categoria STRING, quantidade INT, preco FLOAT
""")

# Executando os UPDATES linha por linha
spark.sql("UPDATE estoque_delta SET categoria = 'Computadores', quantidade = 15, preco = 3500.00 WHERE id = 1")
spark.sql("UPDATE estoque_delta SET categoria = 'Periféricos', quantidade = 50, preco = 120.00 WHERE id = 2")
spark.sql("UPDATE estoque_delta SET categoria = 'Periféricos', quantidade = 30, preco = 250.00 WHERE id = 3")

spark.sql("select * from estoque_delta").show()

### DELETE - Exclusão de Dados
O comando `DELETE FROM` remove um registro específico da tabela. Abaixo, vamos simular que o 'Teclado Mecânico' (id = 3) esgotou ou saiu de linha, deletando-o do estoque.

In [ ]:
spark.sql("DELETE FROM estoque_delta WHERE id = 3")

spark.sql("select * from estoque_delta").show()

### Histórico e Auditoria (Time Travel)
Uma das maiores vantagens do Delta Lake é a manutenção do log de transações. Podemos utilizar o comando `DESCRIBE HISTORY` para auditar tudo o que fizemos na tabela (criação, adição de colunas, inserts, updates e delete).

In [ ]:
from IPython.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

spark.sql('DESCRIBE HISTORY estoque_delta').show(truncate=False)